In [ ]:
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

N, Q, Y = 11, 1, 26
V = 1 + ((17 * N + 7 * Q + Y) % 20)
assert V == 1

K, BUDGET = 30, 135
MIN_COUNT, MAX_COUNT = 3, 8
POP, NSGA_GEN = 50, 104
WEIGHTED_GEN = 20
SEEDS = list(range(3101, 3121))
WEIGHTS = [(0.70, 0.15, 0.15), (0.50, 0.35, 0.15),
           (0.50, 0.15, 0.35), (0.30, 0.50, 0.20),
           (0.30, 0.20, 0.50)]
EVALS = POP * (NSGA_GEN + 1)
assert EVALS == len(WEIGHTS) * POP * (WEIGHTED_GEN + 1)

RESULTS = Path('LR3_variant1_results')
RESULTS.mkdir(exist_ok=True)
print('Variant:', V, 'fitness evaluations per method and seed:', EVALS)

In [ ]:
rng_data = np.random.default_rng(202611)
cost = rng_data.integers(10, 31, K)
effect = 18 + np.rint(1.5 * cost).astype(int) + rng_data.integers(-12, 19, K)
risk = np.clip(np.rint(effect / 6).astype(int) + rng_data.integers(-4, 5, K), 2, 20)

with (RESULTS / 'projects.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'effect', 'cost', 'risk'])
    writer.writerows((i + 1, int(effect[i]), int(cost[i]), int(risk[i])) for i in range(K))

EFFECT_REF = int(sum(sorted(effect, reverse=True)[:MAX_COUNT]))
RISK_REF = MAX_COUNT * int(max(risk))

def values(x):
    return int(x @ effect), int(x @ cost), int(x @ risk)

def objectives(x):
    e, c, r = values(x)
    return (-e, c, r)               # all three are minimized

def feasible(x):
    return MIN_COUNT <= int(sum(x)) <= MAX_COUNT and int(x @ cost) <= BUDGET

def normalized(f):
    return np.array([1 + f[0] / EFFECT_REF, f[1] / BUDGET, f[2] / RISK_REF])

In [ ]:
def repair(x, rng):
    x = np.array(x, dtype=np.int8, copy=True)
    while sum(x) > MAX_COUNT or int(x @ cost) > BUDGET:
        chosen = np.flatnonzero(x)
        x[rng.choice(chosen)] = 0
    while sum(x) < MIN_COUNT:
        options = np.flatnonzero((x == 0) & (cost + int(x @ cost) <= BUDGET))
        if not len(options):
            raise RuntimeError('No feasible project can be added')
        x[rng.choice(options)] = 1
    return x

def random_solution(rng):
    x = np.zeros(K, dtype=np.int8)
    count = int(rng.integers(MIN_COUNT, MAX_COUNT + 1))
    x[rng.choice(K, size=count, replace=False)] = 1
    return repair(x, rng)

def offspring(a, b, rng):
    if rng.random() < 0.9:
        cut = int(rng.integers(1, K))
        child = np.r_[a[:cut], b[cut:]]
    else:
        child = a.copy()
    child = child.copy()
    child[rng.random(K) < 1 / K] ^= 1
    return repair(child, rng)

In [ ]:
def dominates(a, b):
    return (a[0] <= b[0] and a[1] <= b[1] and a[2] <= b[2]
            and (a[0] < b[0] or a[1] < b[1] or a[2] < b[2]))

def nondominated_sort(fitness):
    count = [0] * len(fitness)
    beaten = [[] for _ in fitness]
    first = []
    for i in range(len(fitness)):
        for j in range(i + 1, len(fitness)):
            if dominates(fitness[i], fitness[j]):
                beaten[i].append(j)
                count[j] += 1
            elif dominates(fitness[j], fitness[i]):
                beaten[j].append(i)
                count[i] += 1
        if count[i] == 0:
            first.append(i)
    fronts = [first]
    while fronts[-1]:
        next_front = []
        for i in fronts[-1]:
            for j in beaten[i]:
                count[j] -= 1
                if count[j] == 0:
                    next_front.append(j)
        if not next_front:
            break
        fronts.append(next_front)
    return fronts

def crowding(front, fitness):
    distance = {i: 0.0 for i in front}
    if len(front) < 3:
        return {i: float('inf') for i in front}
    for axis in range(3):
        ordered = sorted(front, key=lambda i: fitness[i][axis])
        span = fitness[ordered[-1]][axis] - fitness[ordered[0]][axis]
        if span == 0:
            continue
        distance[ordered[0]] = distance[ordered[-1]] = float('inf')
        for j in range(1, len(ordered) - 1):
            if np.isfinite(distance[ordered[j]]):
                distance[ordered[j]] += (fitness[ordered[j + 1]][axis]
                                         - fitness[ordered[j - 1]][axis]) / span
    return distance

def rank_crowding(fitness):
    ranks = [0] * len(fitness)
    distances = [0.0] * len(fitness)
    for rank, front in enumerate(nondominated_sort(fitness)):
        cd = crowding(front, fitness)
        for i in front:
            ranks[i], distances[i] = rank, cd[i]
    return ranks, distances

def pareto_unique(population, fitness):
    unique = {tuple(map(int, x)): (np.array(x).copy(), f)
              for x, f in zip(population, fitness)}
    items = list(unique.values())
    front = nondominated_sort([f for _, f in items])[0]
    return [items[i] for i in front]

In [ ]:
def tournament_nsga(ranks, distances, rng):
    a, b = rng.integers(0, len(ranks), size=2)
    if ranks[a] < ranks[b]:
        return int(a)
    if ranks[b] < ranks[a]:
        return int(b)
    if distances[a] > distances[b]:
        return int(a)
    if distances[b] > distances[a]:
        return int(b)
    return int(rng.choice([a, b]))

def nsga2(seed):
    rng = np.random.default_rng(seed)
    pop = [random_solution(rng) for _ in range(POP)]
    fitness = [objectives(x) for x in pop]
    history = []
    for gen in range(NSGA_GEN + 1):
        ranks, distances = rank_crowding(fitness)
        if gen % 10 == 0 or gen == NSGA_GEN:
            history.append((gen, max(-f[0] for f in fitness),
                            len(pareto_unique(pop, fitness))))
        if gen == NSGA_GEN:
            break
        children = []
        while len(children) < POP:
            a = pop[tournament_nsga(ranks, distances, rng)]
            b = pop[tournament_nsga(ranks, distances, rng)]
            children.append(offspring(a, b, rng))
        combined = pop + children
        combined_f = fitness + [objectives(x) for x in children]
        selected = []
        for front in nondominated_sort(combined_f):
            if len(selected) + len(front) <= POP:
                selected.extend(front)
            else:
                distances = crowding(front, combined_f)
                selected.extend(sorted(front, key=lambda i: distances[i], reverse=True)
                                [:POP - len(selected)])
                break
        pop = [combined[i] for i in selected]
        fitness = [combined_f[i] for i in selected]
    return pareto_unique(pop, fitness), history, EVALS

In [ ]:
def weighted_sum(seed):
    final_pop, final_f = [], []
    for wi, weight in enumerate(WEIGHTS):
        rng = np.random.default_rng(seed * 10 + wi)
        pop = [random_solution(rng) for _ in range(POP)]
        f = [objectives(x) for x in pop]
        scores = [float(np.dot(weight, normalized(v))) for v in f]
        for _ in range(WEIGHTED_GEN):
            best = int(np.argmin(scores))
            children = []
            for _ in range(POP):
                i, j = rng.integers(0, POP, size=2)
                a = pop[i if scores[i] < scores[j] else j]
                i, j = rng.integers(0, POP, size=2)
                b = pop[i if scores[i] < scores[j] else j]
                children.append(offspring(a, b, rng))
            child_f = [objectives(x) for x in children]
            child_scores = [float(np.dot(weight, normalized(v))) for v in child_f]
            worst = int(np.argmax(child_scores))
            if scores[best] < child_scores[worst]:
                children[worst] = pop[best].copy()
                child_f[worst] = f[best]
                child_scores[worst] = scores[best]
            pop, f, scores = children, child_f, child_scores
        final_pop.extend(pop)
        final_f.extend(f)
    return pareto_unique(final_pop, final_f), EVALS

In [ ]:
assert dominates((-12, 8, 3), (-10, 9, 4))
assert not dominates((-10, 9, 4), (-12, 8, 3))
assert nondominated_sort([(-10, 10, 10), (-9, 11, 11)])[0] == [0]
rng_test = np.random.default_rng(123)
for _ in range(200):
    a = random_solution(rng_test)
    b = random_solution(rng_test)
    assert feasible(a) and feasible(offspring(a, b, rng_test))

HV_POINTS = np.random.default_rng(99).random((6000, 3))
def volume(front):
    if not front:
        return 0.0
    coords = np.array([normalized(f) for _, f in front])
    return float(np.any(np.all(coords[:, None, :] <= HV_POINTS[None, :, :], axis=2), axis=0).mean())

def save_csv(path, fieldnames, rows):
    with (RESULTS / path).open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

In [ ]:
rows, histories = [], []
nsga_fronts, weighted_fronts = [], []
for seed in SEEDS:
    f_nsga, hist, budget_nsga = nsga2(seed)
    f_weight, budget_weight = weighted_sum(seed)
    assert budget_nsga == budget_weight == EVALS
    nsga_fronts.extend(f_nsga)
    weighted_fronts.extend(f_weight)
    for gen, best_effect, size in hist:
        histories.append(dict(seed=seed, generation=gen,
                              best_effect=best_effect, front_size=size))
    for name, front in [('NSGA-II', f_nsga), ('Weighted sum', f_weight)]:
        assert all(feasible(x) for x, _ in front)
        assert all(not dominates(a[1], b[1])
                   for i, a in enumerate(front) for j, b in enumerate(front) if i != j)
        rows.append(dict(seed=seed, method=name, evaluations=EVALS,
                         front_size=len(front), volume=round(volume(front), 6),
                         max_effect=max(-f[0] for _, f in front),
                         min_cost=min(f[1] for _, f in front),
                         min_risk=min(f[2] for _, f in front)))

save_csv('runs.csv', list(rows[0]), rows)
save_csv('convergence.csv', list(histories[0]), histories)
pooled_nsga = pareto_unique([x for x, _ in nsga_fronts], [f for _, f in nsga_fronts])
pooled_weighted = pareto_unique([x for x, _ in weighted_fronts], [f for _, f in weighted_fronts])

def write_front(path, front):
    result = []
    for x, f in sorted(front, key=lambda p: -p[1][0]):
        result.append(dict(projects=' '.join(str(i+1) for i in np.flatnonzero(x)),
                           count=int(sum(x)), effect=-f[0], cost=f[1], risk=f[2]))
    save_csv(path, list(result[0]), result)
write_front('pareto_nsga.csv', pooled_nsga)
write_front('pareto_weighted.csv', pooled_weighted)

In [ ]:
summary = []
for name in ['NSGA-II', 'Weighted sum']:
    part = [r for r in rows if r['method'] == name]
    for metric in ['volume', 'front_size', 'max_effect', 'min_cost', 'min_risk']:
        a = np.array([r[metric] for r in part], dtype=float)
        summary.append(dict(method=name, metric=metric, minimum=float(a.min()),
                            mean=float(a.mean()), median=float(np.median(a)),
                            std=float(a.std(ddof=1)), maximum=float(a.max())))
save_csv('summary.csv', list(summary[0]), summary)

print('Comparison (20 seeds, equal budgets):')
for name in ['NSGA-II', 'Weighted sum']:
    s = {r['metric']: r for r in summary if r['method'] == name}
    print(f"{name:13s}  volume={s['volume']['mean']:.4f} +/- {s['volume']['std']:.4f}"
          f"  front size={s['front_size']['mean']:.1f}"
          f"  maximum effect={s['max_effect']['mean']:.1f}")

In [ ]:
# Three illustrative portfolios from the aggregate NSGA-II front
sorted_front = sorted(pooled_nsga, key=lambda p: p[1][1])
cheap = sorted_front[0]
high_effect = max(pooled_nsga, key=lambda p: -p[1][0])
center_cost = float(np.median([f[1] for _, f in pooled_nsga]))
intermediate = [p for p in pooled_nsga if p is not cheap and p is not high_effect]
balanced = min(intermediate, key=lambda p: (abs(p[1][1] - center_cost), p[1][2]))
examples = [('Economical', cheap), ('Compromise', balanced), ('Maximum effect', high_effect)]
example_rows = []
for name, (x, f) in examples:
    example_rows.append(dict(type=name, projects=' '.join(str(i + 1) for i in np.flatnonzero(x)),
                             count=int(sum(x)), effect=-f[0], cost=f[1], risk=f[2]))
save_csv('examples.csv', list(example_rows[0]), example_rows)
for r in example_rows:
    print(r)

In [ ]:
# Projections of the Pareto sets, pooled across 20 runs
plt.rcParams['font.size'] = 10
sets = [(pooled_weighted, 'Weighted sum', '#b46a35'),
        (pooled_nsga, 'NSGA-II', '#2771ad')]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for front, name, color in sets:
    eff = [-f[0] for _, f in front]
    cst = [f[1] for _, f in front]
    rsk = [f[2] for _, f in front]
    axes[0].scatter(cst, eff, s=16, alpha=0.65, label=name, color=color)
    axes[1].scatter(rsk, eff, s=16, alpha=0.65, label=name, color=color)
    axes[2].scatter(cst, rsk, s=16, alpha=0.65, label=name, color=color)
axes[0].set(xlabel='Cost', ylabel='Effect')
axes[1].set(xlabel='Risk', ylabel='Effect')
axes[2].set(xlabel='Cost', ylabel='Risk')
for ax in axes:
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8)
fig.suptitle('Pareto set: three two-dimensional projections', fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS / 'pareto_projections.png', dpi=160)
plt.show()

In [ ]:
# NSGA-II convergence across 20 independent seeds
for metric, ylabel, filename in [
    ('best_effect', 'Best effect', 'convergence_effect.png'),
    ('front_size', 'Number of unique Pareto solutions', 'convergence_front.png')]:
    gens = sorted(set(r['generation'] for r in histories))
    traces = [[r[metric] for r in histories if r['seed'] == seed] for seed in SEEDS]
    arr = np.asarray(traces)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(gens, arr.mean(axis=0), color='#2771ad', label='Mean')
    ax.fill_between(gens, arr.min(axis=0), arr.max(axis=0), color='#2771ad', alpha=0.17,
                    label='Min-max across 20 runs')
    ax.set(xlabel='Generation', ylabel=ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(RESULTS / filename, dpi=160)
    plt.show()

fig, ax = plt.subplots(figsize=(7.0, 3.3))
ax.boxplot([[r['volume'] for r in rows if r['method'] == method]
            for method in ['NSGA-II', 'Weighted sum']],
           tick_labels=['NSGA-II', 'Weighted sum'])
ax.set(ylabel='Estimated dominated volume (0..1)',
       title='Front quality over 20 seeds')
ax.grid(alpha=0.25, axis='y')
fig.tight_layout()
fig.savefig(RESULTS / 'volume_boxplot.png', dpi=160)
plt.show()

In [ ]:
# Independent replay check and final data consistency
replay_front, replay_history, replay_evals = nsga2(SEEDS[0])
original = [r for r in rows if r['seed'] == SEEDS[0] and r['method'] == 'NSGA-II'][0]
assert replay_evals == EVALS
assert len(replay_front) == original['front_size']
assert round(volume(replay_front), 6) == original['volume']
assert len(rows) == 40 and len(histories) == 20 * 12
print('Checks passed; results saved to', RESULTS.resolve())